# Interactive Simulation of N₂(g) + 3H₂(g) ⇌ 2NH₃(g) Equilibrium

*This notebook contains an interactive dashboard to simulate the chemical kinetics of the reversible gas-phase reaction $N_2(g) + 3H_2(g) ⇌ 2NH_3(g)$, or the Haber Process. The tool allows for the setting of initial conditions (concentrations, volume, temperature) and the application of real-time, ramped perturbations (volume, injections, and temperature) to quantitatively model and visualise Le Chatelier's principle. This simulation is a step up from the previous one (simulating the equilibria $N_2O_4(g) ⇌ 2NO_2$) as it includes temperature dependence.*

*The simulation is built upon a robust, iterative ODE solver that can handle "stiff" kinetics characteristic of reactions with strong temperature dependence, ensuring a physically accurate results across a wide range of conditions.*

The simulation employs an Object-Oriented Programming (OOP) architecture to model the chemical system. The core principles of this are:
- __Encapsulation:__ the system's state (moles, volume) and the physical laws that govern it (the ODEs) are bundled into a single, self-contained ```ChemicalSystem``` object.
- __State management:__ the state is managed by passing complete ```ChemicalSystem``` objects to ensure data integrity is maintained.

While Le Chatelier's principle can provide a qualitative prediction (e.g., "equilibrium shifts right"), this simulation provides the __quantitative__ answer. It shows by *how much* the concentrations change and *how long* it takes for the new equilibrium to be established.

The __thermodynamic state__ of the system is defined by its temperature, pressure, and composition. For a given temperature, the ratio of the rate constants ($K_c = k_f/k_r$) defines a single, thermodynamically stable endpoint. This equilibrium constant, $K_c$, represents the target that the system is inexorably driven towards. The model visualises this target with a purple ```K_c``` line. The reaction is exothermic ($\Delta H$ = -92 kJ/mol}). By Le Chatelier's principle, low temperatures favour the forward reaction and high equilibrium yields of ammonia.

While thermodynamics determines *where* the equilibrium lies, it says nothing about *how long* it will take to get there. That is the domain of __kinetics__. The speed of the journey is governed by the absolute magnitudes of the forward and reverse rate constants, $k_f$ and $k_r$. These are determined by:
- The activation energies ($E_a$): these are the fundamental barriers to reaction, representing the "steepness of the hill" that the molecules must climb. A high $E_a$ leads to a slow rate.
- The temperature (T): this provides the energy for molecules to overcome the activation barrier. As temperature increases, the rate constants increase exponentially, as described by the Arrhenius equation.

This simulation illustrates the distinction between thermodynamics and kinetics. For example, cooling the Haber process from 700 K to 550 K is thermodynamically favourable (the new $K_c$ is much larger). However, the kinetic reality is that the reaction becomes significantly *slower* because fewer molecules have the energy to overcome the activation barrier. 

## 1. Imports

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import display
import ipywidgets as widgets
import pandas as pd
import copy

## 2. Model Configuration
This defines the physical and chemical constants of the simulation.

In [2]:
# physical constants...
Ea_f_J_mol = 85000 # J mol^-1, forwards activation energy
Ea_r_J_mol = 177000 # J mol^-1, reverse activation energy
T_REF_K = 900 # Kelvin, reference temperature
K_F_REF = 0.1 # dm^9 mol^-3 s^-1, forward rate constant, k_f, at reference temperature
K_R_REF = 2.5 # dm^3 mol^-1 s^-1, reverse rate constant, k_r, at reference temperature
ENTHALPY_CHANGE = -92000 #J mol^-1

R = 8.314 # universal gas constant in J K^-1 mol^-1
R_ATM_L = 0.08206 # universal gas constant in L atm K^-1 mol^-1, for pressure calculations

# tolerances for the equilibrium-finding algorithm...
RATE_TOLERANCE = 1e-5
SLOPE_TOLERANCE = 1e-3
QC_KC_TOLERANCE = 1e-3

# van der Waals Constants for the 3 species
# Units: a in dm^6 atm mol^-2, b in dm^3 mol^-1
# Source: CRC Handbook of Chemistry and Physics, via Wikipedia. Converted a values from bar to atm.
VDW_CONSTANTS = {
    'N2':  {'a': 1.352, 'b': 0.0387},  # calculated: 1.370 * 0.9869
    'H2':  {'a': 0.244, 'b': 0.0266},  # calculated: 0.2476 * 0.9869
    'NH3': {'a': 4.170, 'b': 0.0371}}   # calculated: 4.225 * 0.9869

### Physical Constants and Sources

For a reaction like the Haber process, there is no single, universally agreed-upon set of "right" kinetic parameters, because the reaction is catalysed. The measured activation energy and pre-exponential factor are not properties of the gas-phase molecules themselves, but properties of the entire system, dominated by the nature of the catalyst. The values depend intensely on the catalyst, the promoters, etc. Therefore, these values are chosen to satisfy 3 non-negotiable constraints:

1. _Thermodynamic consistency:_ ΔH = Ea_f - Ea_r.
   The enthalpy change of the reaction is ($\Delta H^o$) = -92.2 kJ/mol (Atkins, P. W., & de Paula, J. (2014). Physical Chemistry. Oxford University Press). The values for the forward and reverse activation energies satisfy the equation and anchors the model.
2. _Plausible magnitude:_ the individual activation energies are in the correct order of magnitude for a real, catalysed industrial process. They are not the values for a slow, uncatalysed gas-phase reaction (which would be much higher, >200-300 kJ/mol), nor are they trivially small. They are representative of the energy barriers on a real catalytic surface.
3. _Computational traceability:_ the reference rate constants ($k_{f,ref}, k_{r, ref}$) are chosen such that when combined with these activation energies, the system reaches equilibrium on a timescale of seconds to minutes in the simulation.
   
These kinetic parameters are the most scientifically sound approach for our model's purpose, which is to model the principles of equilibrium.

The Arrhenius equation mathematically describes the observation that most reaction rates increase exponentially with temperature. This is because a higher temperature leads to a greater fraction of reactant molecules possessing energy equal to or greater than the activation energy ($E_a$) leading to more frequent successful collisions.

Because the model calculates both $k_f$ and $k_r$ using their respective Arrhenius equations based on the reference data, the equilibrium constant $K_c = k_f/k_r$ will *automatically* and correctly obey the van 't Hoff equation, which describes how $K_c$ changes with temperature. This provides a powerful, built-in thermodynamic consistency check on the kinetic model.

The activation energies defined above are not for the uncatalysed gas-phase reaction, which creates a barrier too high to overcome at reasonable temperatures. These parameters model the catalysed pathway (typically using an iron substrate), where the mechanism involves adsorption of $N_2$ and $H_2$ on the catalyst's surface, weakening the bonds and lowering the activation energy. The simulation assumes the catalyst surface area is constant.

The Van der Waals constants are needed for the calculations of pressure according to the Van der Waals equation. The $a$ values taken from the source were in $L^2*bar*mol^{-2}$ and was converted to $L^2*atm*mol^{-2}$ using the conversion: $1 bar ≈ 0.986923 atm$. Observing the $a$ and $b$ values, we see that:
- $a$ (Attraction): ammonia (4.17) is nearly 20 times higher than hydrogen (0.24). This is because ammonia is a polar molecule with hydrogen-bonding, resulting in stronger attractive forces between molecules of ammonia. Nitrogen has a larger $a$ than hydrogen because of its larger electron cloud which makes it more polarisable than hydrogen.
- $b$ (Volume): ammonia (0.037) is larger than hydrogen (0.027) but similar to nitrogen (0.039).

## 3. Simulation Engine
The core of the simulation is a system of ODEs. It takes in the current state (in moles) and system volume, calculates the corresponding concentrations, and returns the rate of change of moles ('dn/dt') for each species. An early version of this model defined the system's state purely by the concentrations of the chemical species. However, to model Le Chatelier's principle for pressure and volume changes, it was essential to re-engineer the simulation to run on more fundamental quantities. This model's state vector therefore tracks the number of moles (n) of each species, while the system volume (V) is treated as a separate, independent variable.
- Forward Reaction ($N_2 + 3H_2 -> 2NH_3$): this is first-order with respect to $N_2$ and third-order with respect to $H_2$.
      Forward Rate = k_f * $[N_2]$ * $[H_2]^3$
- Reverse Reaction ($2NH_3 -> N_2 + 3H_2$): this is second-order with respect to $NH_3$:
      Reverse Rate = k_r * $[NH_3]^2$

The simulation is built upon __2 primary classes:__ ```ChemicalSystem``` and ```PerturbationSimulation```.

### The ```ChemicalSystem``` Class

This is the fundamental data structure of the simulation. 
- __Attributes:__ an instance of ```ChemicalSystem``` encapsulates the intrinsic properties and current state of the reaction: number of moles of each species, system volume, and forward/reverse rate constants.
- __Methods:__ it contains the methods that define its behaviour. The ```_ode_system``` method provides the mathematical model of the reaction kinetics for the numerical solver. The ```run_simulation``` method orchestrates the integration of this system over time, and the ```_process_results``` method transforms the raw numerical output into chemically meaningful data, such as concentrations, reaction rates, and equilibrium constants.

### The Equilibrium-Finding Algorithm

A key challenge in this simulation is to robustly determine the time at which the system reaches equilibrium, as this must work for both near-instantaneous and slow, asymptotic reactions. For example, a simple approach of waiting for the net reaction rate to fall below a threshold can fail due to numerical noise in very fast reactions.
The algorithm implemented here defines equilibrium as the point where 3 conditions are met simultaneously:
1. The net rate of reaction is negligibly small, checking if the forward and reverse reaction rates have become equal.
   
   ```np.abs(net_rates_N2) < RATE_TOLERANCE```
2. The system has become stable, meaning the rate of reaction is no longer changing significantly.
   
   ```np.abs(np.diff(net_rates_N2)) < SLOPE_TOLERANCE```
3. The reaction quotient ($Q_c$) must have converged to the theoretical equilibrium constant ($K_c = k_f/k_r$).
   
   $\left|\frac{Q_c - K_c}{K_c}\right|$ < ```QC_KC_TOLERANCE```
The first time point where both conditions are true is considered the equilibrium time. A technical step, np.pad, is used to align the rate and rate-delta arrays for direct comparison.

Ideally, equilibrium is defined as the state where the Gibbs free energy of the system is minimised. The thermodynamic driving force is given by:
    $\Delta _rG = \Delta _rG^o + RT ln(Q_c)$
    
As the system approaches equilibrium, $Q_c$ -> $K_c$, causing $\Delta _rG$ -> 0. 

### Advanced Numerical Methods

An early limitation of this model was its reliance on a fixed simulation time (```MAX_T```). For reactions with temperature-dependent kinetics, this is a critical flaw. A time sufficient for a reaction at 800 K might be orders of magnitude too short for the same reaction at 500 K, leading to a failure to reach equilibrium. To overcome this, the simulation engine was fundamentally re-engineered from a single-pass solver to an intelligent, iterative process.

The ```run_simulation``` method now employs an iterative algorithm to actively search for equilibrium. The process is as follows:
- The simulation is run in an initial, short time "chunk" (e.g., 100s).
- After the chunk is complete, the data is processed to check if a robust, three-factor equilibrium condition has been met.
- If equilibrium is not found, the end-state of the completed chunk becomes the initial state for a new, longer chunk, and the process repeats.
- This continues until equilibrium is detected, ensuring the simulation runs for precisely as long as is necessary for the given physical conditions. This eliminates user guesswork and makes the model robust across vastly different timescales.

### Real Gases
__Failure of the Ideal Gas Law__

Another early limitation of this model was its use of the ideal gas approximation. At the high pressures typical of the Haber process, the Ideal Gas Law ($PV = nRT$) breaks down. It assumes molecules are point masses with no volume and no intermolecular forces.
1. Molecular volume ($b$): at high pressures, molecules are crowded. The "free volume" available for movement is less than the container volume ($V - nb$), tending to *increase* pressure.
2. Intermolecular forces ($a$): molecules attract each other (van der Waals forces), striking the container walls with less force, tending to *decrease* pressure.

__Van der Waals Equation__

To model this, the simulation uses the Van der Waals equation of state with species-specific constants ($a$ and $b$):
$$(P + a(n/V)^2)(V - nb) = nRT$$

Where $P$ is pressure, $V$ is volume, $n$ is the number of moles, $T$ is temperature, $R$ is the gas constant, $a$ is a measure of intermolecular attraction, and $b$ is the volume excluded by a mole of particles.

The simulation employs this equation to calculate the "Real Pressure" ($P_{real}$) and compare it to the ideal pressure.

__Mixing Rules__

Since the system contains a dynamic mixture of $N_2$, $H_2$, and $NH_3$, the constants $a$ and $b$ are calculated dynamically at every time step using mixing rules:
- $ a_{\text{mix}} = \left( \sum_i x_i \sqrt{a_i} \right)^2 $ (attraction is pairwise, requiring the geometric mean)
- $b_{\text{mix}} = \sum_i x_i b_i$ (volume is additive)

These calculations are handled by the ```calculate_pressure``` method.

__$K_p$ vs $K_c$__

Chemical equilibrium can be defined using concentrations ($K_c$) or partial pressures ($K_p$). 

The simulation calculates $K_p$ using partial pressures. Note that $K_c$ and $K_p$ differ significantly due to the change in moles ($\Delta n = -2$):
    $K_p = K_c(RT)^{\Delta n}$
    
Which simplifies to:
    $K_p = K_c/(RT)^2$
    
Since $(RT)^2$ is large at reaction temperatures, $K_p$ is order of magnitudes smaller than $K_c$. On the simulation dashboard, the "Thermodynamics" axis uses a Logarithmic Scale to visualize both constants simultaneously, clearly showing the thermodynamic gap $(RT)^{-2}$ between them.

__Stress Testing__

During extreme stress testing (high moles, near-zero volume), the simulation exhibited a mathematical breakdown where the calculated Real Pressure flipped from very high positive values to negative values.

This was because V starting approaching nb or became less than nb, causing the denominator ($V - nb$) to become negative, and results in a physically impossible negative pressure calculation.

In an example test case, nearly 20 moles was compressed into $0.05 dm^3$, so the excluded volume of the atoms ($nb ≈ 0.62 dm^3$) exceeded the container volume ($V = 0.05 dm^3$). Mathematically, this implies the atoms are overlapping, violating the Pauli Exclusion Principle.

For that test case, the graph showed a sharp downward spike (a dip) in $Q_p$ just before the pressure flipped.
1. Since $\Delta n = -2$, $Q_p$ is proportional to $P^{-2}$.
2. As the $V$ approached $nb$, the pressure approached infinity ($P -> \infty$).
3. Consequently, the quotient approached zero ($Q_p \propto 1/\infty -> 0$).
4. On the logarithmic scale used by the dashboard, approaching zero appears as a vertical drop toward $-\infty$.

This shows that the model is valid only for gas densities where the free volume is positive ($V > nb$). To prevent numerical errors in future runs, a safety clamp was added to the ```calculate_pressure``` method to return an asymptotic pressure value as $V$ approaches $nb$ rather than allowing the sign flip.

This issue exposes 2 key limitations of the model:
- The model assumes the species remain in the gas phase at all densities. In reality, before reaching $V ≈ nb$, the gas would undergo a phase transition.
- The Van der Waals model treats atoms as hard spheres that cannot overlap. It does not account for the soft-potential repulsion or quantum mechanical effects that dominate at such extreme densities.

In [3]:
class ChemicalSystem:
    """Represents the N₂(g) + 3H₂(g) ⇌ 2NH₃(g) chemical system.
    This class encapsulates the state of the system (moles, volume) and its kinetic parameters (rate constants),
    providing the methods to model its dynamic behaviour."""
    
    def __init__(self, initial_moles_N2, initial_moles_H2, initial_moles_NH3, V, T):
        """Initialises the chemical system with its starting conditions."""

        self.n_N2 = initial_moles_N2
        self.n_H2 = initial_moles_H2
        self.n_NH3 = initial_moles_NH3
        self.volume = V
        self.T = T
        self.Ea_f = Ea_f_J_mol
        self.Ea_r = Ea_r_J_mol
        self.T_ref = T_REF_K
        self.k_f_ref = K_F_REF
        self.k_r_ref = K_R_REF
        self.y0 = [self.n_N2, self.n_H2, self.n_NH3] # initial state vector for the ODE solver

        self.k_f = None
        self.k_r = None
        self.solution = None # placeholder for the simulation results
        self.results = None # placeholder for the processed results dictionary

        self._update_rate_constants()

    def _ode_system(self, t, y):
        """This defines the system of Ordinary Differential Equations (ODEs) for the reaction. This method calculates the rate of change of moles for
        each species based on the current state of the system. It is intended for usewith an ODE solver like scipy.integrate.solve_ivp.

        It returns a list containing the rate of change for each species [dn_N2/dt, dn_H2/dt, dn_NH3/dt]."""
    
        moles_N2, moles_H2, moles_NH3 = y[0], y[1], y[2]  # unpacking the state vector...

        # delegate the rate calculation to a centralised method...
        return self.get_rates_at_state(moles_N2, moles_H2, moles_NH3, self.volume, self.T)

    def get_rates_at_state(self, moles_N2, moles_H2, moles_NH3, V, T):
        """This is the centralised method that calculates derivatives given a specific set of molar amounts, volume, and temperature.

        This method calculates rate constants dynamically based on the provided T, ensuring kinetics are accurate even if T changes during a
        perturbation."""
        
        # 1. Dynamically calculate k values for the specific T provided...
        k_f_dynamic = self.__class__._calculate_arrhenius_k(T, self.T_ref, self.Ea_f, self.k_f_ref)
        k_r_dynamic = self.__class__._calculate_arrhenius_k(T, self.T_ref, self.Ea_r, self.k_r_ref)

        # 2. Calculate concentrations based on the specific V provided...
        conc_N2 = moles_N2 / V
        conc_H2 = moles_H2 / V
        conc_NH3 = moles_NH3 / V

        # 3. Calculate rates using the rate laws...
        # Rate = k * [Concentration]^Order
        forward_rate = k_f_dynamic * conc_N2 * conc_H2**3
        reverse_rate = k_r_dynamic * conc_NH3**2

        # 4. Calculate derivatives (dn/dt) based on stoichiometry...
        # Rate is in mol dm^-3 s^-1. Multiply by V to get mol s^-1.
        dn_N2_dt = V * (-forward_rate + reverse_rate)
        dn_H2_dt = 3 * dn_N2_dt
        dn_NH3_dt = -2 * dn_N2_dt

        return [dn_N2_dt, dn_H2_dt, dn_NH3_dt]
        
    def run_simulation(self):
        """This runs the simulation by integrating the ODE system. This method calls the ODE solver to compute
        the evolution of the system over time and stores the result within the object. 
        max_t (float) represents the total time duration for the simulation in seconds."""

        y0 = self.y0 # initialise
        solutions_list = []
        time_offset = 0.0 # tracks the cumulative time
        current_chunk_duration = 100.0 # a reasonable starting guess
        max_iterations = 15 # maximum number of iterations to prevent an infinite loop
        equilibrium_found = False
        for i in range(max_iterations):
            t_span_chunk = (time_offset, time_offset + current_chunk_duration) # the timespan for the chunk
            t_eval_chunk = np.linspace(t_span_chunk[0], t_span_chunk[1], 5000) # creates the time array
            
            # calling the solver using the object's own ODE system...

            chunk_solution = solve_ivp(fun = self._ode_system, t_span = t_span_chunk, y0 = y0, dense_output = True, method='LSODA', rtol=1e-6, atol=1e-9, t_eval=t_eval_chunk)
            solutions_list.append(chunk_solution)
            y0 = chunk_solution.y[:, -1]
            time_offset = chunk_solution.t[-1]
            combined_t = np.concatenate([s.t for s in solutions_list])
            combined_y = np.concatenate([s.y for s in solutions_list], axis=1)
            self.solution = {'t': combined_t, 'y': combined_y} # temporarily set self.solution
            eq_time = self._process_results(check_only=True)

            if eq_time is not None:
                print(f"Equilibrium found at t = {eq_time:.4f} s.")
                equilibrium_found = True
                break # exit the for loop

            print(f"Equilibrium not found after {time_offset:.1f}s. Extending simulation...")
            current_chunk_duration *= 10

        combined_t = np.concatenate([s.t for s in solutions_list])
        combined_y = np.concatenate([s.y for s in solutions_list], axis=1)
        # automatically call the processor...
        self.solution = {'t': combined_t, 'y': combined_y}
        self._process_results()

    def _process_results(self, check_only = False):
        """This processes the raw output from the ODE solver into a structured dictionary containing chemically meaningful data
        such as concentrations, rates, Kc, etc."""

        if self.solution is None:
            print("Error: simulation has not been run yet. Cannot process results.")
            return

        # 1. Data extraction...
        time = self.solution['t']

        moles_vs_time = self.solution['y']
        moles_N2_vs_time = moles_vs_time[0]
        moles_H2_vs_time = moles_vs_time[1]
        moles_NH3_vs_time = moles_vs_time[2]

        # 2. Calculations...
        conc_N2_vs_time = moles_N2_vs_time / self.volume
        conc_H2_vs_time = moles_H2_vs_time / self.volume
        conc_NH3_vs_time = moles_NH3_vs_time / self.volume
        
        # we pass the full arrays to the engine, which returns [dn_N2/dt, dn_H2/dt, dn_NH3/dt]
        # then divide dn/dt by volume to get rate in mol dm^-3 s^-1
        dn_dt_arrays = self.get_rates_at_state(moles_N2_vs_time, moles_H2_vs_time, moles_NH3_vs_time, self.volume, self.T)
        net_rates_N2 = dn_dt_arrays[0] / self.volume

        k_f_val = self.__class__._calculate_arrhenius_k(self.T, self.T_ref, self.Ea_f, self.k_f_ref)
        k_r_val = self.__class__._calculate_arrhenius_k(self.T, self.T_ref, self.Ea_r, self.k_r_ref)
        
        forward_rates_vs_time = k_f_val * conc_N2_vs_time * conc_H2_vs_time**3
        reverse_rates_vs_time = k_r_val * conc_NH3_vs_time**2
        
        Qc_vs_time = np.divide(
            conc_NH3_vs_time**2, (conc_N2_vs_time * conc_H2_vs_time**3),
            out=np.full_like(conc_N2_vs_time, np.nan),
            where=((conc_N2_vs_time * conc_H2_vs_time**3) != 0))

        # 3. Compute equilibrium values...
        conc_N2_eq = conc_N2_vs_time[-1]
        conc_H2_eq = conc_H2_vs_time[-1]
        conc_NH3_eq = conc_NH3_vs_time[-1]
        moles_N2_eq = moles_N2_vs_time[-1]
        moles_H2_eq = moles_H2_vs_time[-1]
        moles_NH3_eq = moles_NH3_vs_time[-1]
        n_total_eq = moles_N2_eq + moles_H2_eq + moles_NH3_eq
        n_total_vs_time = np.sum(moles_vs_time, axis=0) 
        P_ideal = (n_total_vs_time * R_ATM_L * self.T) / self.volume
        P_ideal_eq = P_ideal[-1]
        P_real = self.calculate_pressure(n_N2 = moles_N2_vs_time, n_H2 = moles_H2_vs_time, n_NH3 = moles_NH3_vs_time, V = self.volume, T = self.T)
        P_real_eq = P_real[-1]
        
        Kc = (conc_NH3_eq**2) / (conc_N2_eq * conc_H2_eq**3) if (conc_N2_eq * conc_H2_eq**3) != 0 else None

        with np.errstate(divide='ignore', invalid='ignore'):
            # mole fractions...
            x_N2 = np.divide(moles_N2_vs_time, n_total_vs_time, where=n_total_vs_time!=0)
            x_H2 = np.divide(moles_H2_vs_time, n_total_vs_time, where=n_total_vs_time!=0)
            x_NH3 = np.divide(moles_NH3_vs_time, n_total_vs_time, where=n_total_vs_time!=0)

            # partial Pressures (Dalton's Law on Real Pressure)...
            pp_N2 = x_N2 * P_real
            pp_H2 = x_H2 * P_real
            pp_NH3 = x_NH3 * P_real

            # Qp calculation:
            Qp_vs_time = np.divide(
                pp_NH3**2, (pp_N2 * pp_H2**3),
                out=np.full_like(P_real, np.nan),
                where=((pp_N2 * pp_H2**3) != 0))

        # Kp calculation, Kp = Kc * (RT)^delta_n, delta_n = -2.
        k_f_val = self.__class__._calculate_arrhenius_k(self.T, self.T_ref, self.Ea_f, self.k_f_ref)
        k_r_val = self.__class__._calculate_arrhenius_k(self.T, self.T_ref, self.Ea_r, self.k_r_ref)
        Kc_theoretical = k_f_val / k_r_val
        
        # Kp is constant for isothermal, but we calculate as array for compatibility...
        Kp_val = Kc_theoretical * (R_ATM_L * self.T)**(-2)
        Kp_vs_time = np.full_like(time, Kp_val)

        # finding equilibrium time...
        
        rate_deltas = np.diff(net_rates_N2)
        rate_deltas_padded = np.pad(rate_deltas, (1, 0), 'constant', constant_values = 1)

        n_total_vs_time = np.sum(moles_vs_time, axis=0)
        is_depleted = n_total_vs_time < 1e-9 # this checks for if total moles is zero

        eq_indices = np.where(
            (np.abs(net_rates_N2) < RATE_TOLERANCE) &
            (np.abs(rate_deltas_padded) < SLOPE_TOLERANCE) &
            (
                (np.abs(Qc_vs_time - Kc_theoretical) / Kc_theoretical < QC_KC_TOLERANCE) | 
                is_depleted))[0]
        
        if eq_indices.size > 0:
            equilibrium_time = time[eq_indices[0]]
        else:
            equilibrium_time = time[-1] # default to the last value if equilibrium is not found
            
        if eq_indices.size > 0:
            # define a view window that is 20% longer than the time to reach equilibrium
            trunc_time = equilibrium_time * 1.2
            
            # find the first index in the time array that is beyond this window
            end_indices = np.where(time >= trunc_time)[0]
            
            # if such an index exists, truncate all time-dependent arrays to that length...
            if end_indices.size > 0:
                trunc_index = end_indices[0]
                
                time = time[:trunc_index]
                moles_N2_vs_time = moles_N2_vs_time[:trunc_index]
                moles_H2_vs_time = moles_H2_vs_time[:trunc_index]
                moles_NH3_vs_time = moles_NH3_vs_time[:trunc_index]
                conc_N2_vs_time = conc_N2_vs_time[:trunc_index]
                conc_H2_vs_time = conc_H2_vs_time[:trunc_index]
                conc_NH3_vs_time = conc_NH3_vs_time[:trunc_index]
                net_rates_N2 = net_rates_N2[:trunc_index]
                forward_rates_vs_time = forward_rates_vs_time[:trunc_index]
                reverse_rates_vs_time = reverse_rates_vs_time[:trunc_index]
                Qc_vs_time = Qc_vs_time[:trunc_index]
                P_ideal = P_ideal[:trunc_index]
                P_real = P_real[:trunc_index]
                Qp_vs_time = Qp_vs_time[:trunc_index]
                Kp_vs_time = Kp_vs_time[:trunc_index]

                
        if check_only:
            if eq_indices.size > 0:
                return equilibrium_time
            else: return None

        print(f"Theoretical Kc based on k_f and k_r is {Kc_theoretical}.") # prints Kc_theoretical for reference
                
        # 4. Store the results in a dictionary...
        self.results = {
            "time": time,
            "initial_conditions": {"V": self.volume, "T": self.T, "n_N2": self.n_N2, "n_H2": self.n_H2, "n_NH3": self.n_NH3},
            "results": {
                "moles_N2": moles_N2_vs_time, "moles_H2": moles_H2_vs_time, "moles_NH3": moles_NH3_vs_time,
                "conc_N2": conc_N2_vs_time, "conc_H2": conc_H2_vs_time, "conc_NH3": conc_NH3_vs_time,
                "rates_N2": net_rates_N2, "forward_rates": forward_rates_vs_time, "reverse_rates": reverse_rates_vs_time,
                "P_ideal": P_ideal, "P_real": P_real, "Qp_vs_time": Qp_vs_time, "Kp_vs_time": Kp_vs_time, "Qc_vs_time": Qc_vs_time},
            "equilibrium": {"time": equilibrium_time, "T": self.T,
                "conc_N2": conc_N2_eq, "conc_H2": conc_H2_eq, "conc_NH3": conc_NH3_eq,
                "P_ideal": P_ideal_eq, "P_real": P_real_eq, "Kc": Kc, "Kp": Kp_val}}

    def _update_rate_constants(self):
        """Updates the object's rate constants based on its temperature."""
        # finding the forward rate constant, k_f, using the Arrhenius equation...
        self.k_f = self.__class__._calculate_arrhenius_k(self.T, self.T_ref, self.Ea_f, self.k_f_ref)

        # finding the reverse rate constant, k_r, using the Arrhenius equation...
        self.k_r = self.__class__._calculate_arrhenius_k(self.T, self.T_ref, self.Ea_r, self.k_r_ref)

    def calculate_pressure(self, n_N2, n_H2, n_NH3, V, T):
        """Calculates pressure based on the Van der Waals equation for real gases."""
        n_total = n_N2 + n_H2 + n_NH3 # calculates the total number of moles

        # calculating the mole fraction of each species...
        x_N2 = n_N2 / n_total
        x_H2 = n_H2 / n_total
        x_NH3 = n_NH3 / n_total

        # extracting values from the dictionaries...
        a_N2, a_H2, a_NH3 = [VDW_CONSTANTS[gas]['a'] for gas in ('N2', 'H2', 'NH3')]
        b_N2, b_H2, b_NH3 = [VDW_CONSTANTS[gas]['b'] for gas in ('N2', 'H2', 'NH3')]

        with np.errstate(divide='ignore', invalid='ignore'):
             x_N2 = np.divide(n_N2, n_total, where=n_total!=0)
             x_H2 = np.divide(n_H2, n_total, where=n_total!=0)
             x_NH3 = np.divide(n_NH3, n_total, where=n_total!=0)

        # use the mixing rules...
        a_mix = ((x_N2 * np.sqrt(a_N2)) + (x_H2 * np.sqrt(a_H2)) + (x_NH3 * np.sqrt(a_NH3)))**2
        b_mix = x_N2 * b_N2 + x_H2 * b_H2 + x_NH3 * b_NH3

        nb = n_total * b_mix

        # check: if V is extremely closer to or less than nb, the physics breaks
        V_free = V - nb # this is the "free volume"

        if isinstance(V, np.ndarray) or isinstance(nb, np.ndarray):
            # create a mask where the volume is physically impossible
            mask_impossible = V_free <= 1e-6 # 1e-6 safety margin
            
            # to prevent crash, we set V_free to a tiny positive number where it's impossible
            # this causes Pressure to explode to +Infinity (correct behavior) rather than flip negative
            V_free = np.maximum(V_free, 1e-6) 
            
            P_ideal_term = (n_total * R_ATM_L * T) / V_free
            P_real = P_ideal_term - ((a_mix * n_total**2) / V**2)

        else:
            if V_free <= 1e-6:
                # return a massive pressure to indicate failure, or handle gracefully
                return 9.99e9 # "Infinite" pressure
            else:
                 P_real = ((n_total * R_ATM_L * T) / V_free) - ((a_mix * n_total**2) / V**2)
                 
        return P_real
        
    @staticmethod # static method that does not depend on the state of any particular object(self)
    def _calculate_arrhenius_k(T, T_ref, Ea, k_ref):
        """This calculates a rate constant k at temperature T using the Arrhenius equation. This static method is vectorised to handle arrays."""
        T_array = np.asarray(T) # this ensures the input is a NumPy array 
        
        # we must add a check to prevent division by 0 if T is 0 K...
        with np.errstate(divide='ignore'): # suppress warning for 1/0, np.where handles it
            result = np.where(
                T_array <= 0,
                0.0, # value if condition is True
                k_ref * np.exp(-(Ea / R) * ((1 / T_array) - (1 / T_ref)))) # value if condition is False
        return result

### The ```PerturbationSimulation``` Class

This class is designed to handle the complex, 3-stage workflow of applying a stress to a system at equilibrium.
- __Composition:__ this class uses composition; it holds a fully-formed ```ChemicalSystem object``` as an attribute (```self.baseline_system```). The ```PerturbationSimulation``` acts *upon* the chemical system.
- __Orchestration:__ its primary method, ```run_perturbation```, executes the full sequence: it extracts the pre-perturbation history from the baseline system (Stage 1), selects the appropriate mathematical model for the perturbation window and simulates the stress (Stage 2), and finally, simulates the system's relaxation to a new equilibrium (Stage 3). It is responsible for concatenating the data from these distinct phases into a single, coherent time-series.

This simulation quantitatively visualises Le Chatelier's principle by tracking the relationship between $Q_c$ and $K_c$ when a "stress" (perturbation) is applied. 

__The Ramped Volume Model...__
    This takes the interpolation parameters as arguments. This function is used only for the solve_ivp call in stage 2, the stage where the volume is a function of time.

__The Injection Model...__
    This models a gradual injection of a reactant/product. The total rate of change in the number of moles for a given chemical species (dn/dt) is the sum of the change from the internal reaction and any change from an external source. 
dn/dt_total = dn/dt_reaction + dn/dt_injection. This is used for stage 2 of the simulation. Stage 1 (pre-injection) and stage 3 (post-injection) will revert to the original reversible_model.

A concentration/volume stress changes $Q_c$ while $K_c$ remains constant. The system reacts to restore the ratio $Q_c = K_c$.

__The Ramped Temperature Model...__
    This is used for temperature perturbations. $K_c$ changes (according to the van 't Hoff equation) while $Q_c$ initially remains constant. The system reacts because the "target" has moved.

In [4]:
class PerturbationSimulation:
    """This manages the three-stage process of applying a perturbation to a baseline ChemicalSystem and simulating the result.

    This class uses composition, holding a ChemicalSystem instance to represent the initial state. Its primary role is to orchestrate the simulation
    before, during, and after a defined stress is applied."""
    
    def __init__(self, baseline_system, perturbation_type, t_start, t_end,
                 V_end=None, T_end=None, injection_rate_N2=0.0, injection_rate_H2=0.0, injection_rate_NH3=0.0):
        """This initialises the perturbation process manager.
        Args:
            baseline_system (ChemicalSystem): the fully simulated system object to be perturbed.
            perturbation_type (str): the type of stress, either 'volume' or 'injection'.
            t_start, t_end (floats): the start and end times of the perturbation window.
            V_end (float, optional): the target volume for a 'volume' perturbation. Defaults to None.
            injection_rate_N2, injection_rate_H2, injection_rate_NH3 (float, optional): moles/s for an 'injection' perturbation. Defaults to 0.0."""
        
        # store all configuration parameters...
        self.baseline_system = baseline_system
        self.perturbation_type = perturbation_type
        self.t_start = t_start
        self.t_end = t_end

        # store perturbation-specific parameters...
        self.V_start = baseline_system.volume
        self.V_end = V_end
        self.T_end = T_end
        self.injection_rate_N2 = injection_rate_N2
        self.injection_rate_H2 = injection_rate_H2
        self.injection_rate_NH3 = injection_rate_NH3

        self.results = None # placeholders for results

    def _ramped_volume_model(self, t, y):
        """This is an internal ODE system for the volume perturbation stage (Stage 2). It returns a list which is the rate of change of all species.
        Args:
            t (float): current time from the solver.
            y (list): current state vector [moles_N2, moles_H2, moles_NH3]."""
        
        # interpolate to find the volume at the exact time t...
        V_at_t = np.interp(t, [self.t_start, self.t_end], [self.V_start, self.V_end])
        T_constant = self.baseline_system.T

        return self.baseline_system.get_rates_at_state(y[0], y[1], y[2], V_at_t, T_constant)

    def _ramped_temperature_model(self, t, y):
        """This is an internal ODE system for the temperature perturbation stage (Stage 2). It returns a list which is the rate of change of all species."""

        V_constant = self.baseline_system.volume
        # interpolate to find the temperature at the exact time t...
        T_at_t = np.interp(t, [self.t_start, self.t_end], [self.baseline_system.T, self.T_end])

        # ask ChemicalSystem for the rates at these specific conditions (ChemicalSystem will handle the Arrhenius calculations internally)
        return self.baseline_system.get_rates_at_state(y[0], y[1], y[2], V_constant, T_at_t)

    def _injection_model(self, t, y):
        """This is another internal ODE system for the species injection stage (Stage 2). It returns a list of the rate of change of both species.
        Takes in the same arguments as the _ramped_volume_model."""
        
        moles_N2, moles_H2, moles_NH3 = y[0], y[1], y[2]
        V_constant = self.baseline_system.volume # volume is constant during injection
        T_constant = self.baseline_system.T

        dn_dt_from_reaction = self.baseline_system.get_rates_at_state(moles_N2, moles_H2, moles_NH3, V_constant, T_constant)

        # add the external injection rate...
        total_dn_N2_dt = dn_dt_from_reaction[0] + self.injection_rate_N2
        total_dn_H2_dt = dn_dt_from_reaction[1] + self.injection_rate_H2
        total_dn_NH3_dt = dn_dt_from_reaction[2] + self.injection_rate_NH3
        
        # prevent moles from becoming negative...
        if moles_N2 <= 0 and total_dn_N2_dt < 0:
            total_dn_N2_dt = 0
        if moles_H2 <= 0 and total_dn_H2_dt < 0:
            total_dn_H2_dt = 0
        if moles_NH3 <= 0 and total_dn_NH3_dt < 0:
            total_dn_NH3_dt = 0

        return [total_dn_N2_dt, total_dn_H2_dt, total_dn_NH3_dt]

    def run_perturbation(self):
        """This orchestrates the full 3-stage perturbation simulation. This executes the pre-perturbation, during perturbation, and post-perturbation
        stages, then stitches the results together into a single, coherent dataset."""

        # Stage 1: Pre-Perturbation (Data from Baseline)
        baseline_results = self.baseline_system.results
        baseline_time = baseline_results['time']
        T_start = self.baseline_system.T

        # find the index in the baseline data corresponding to the perturbation start time...
        perturb_index = np.argmin(np.abs(baseline_time - self.t_start))

        # slice all baseline arrays to get Stage 1 data...
        stage1_time = baseline_time[:perturb_index + 1]
        stage1_moles_N2 = baseline_results['results']['moles_N2'][:perturb_index + 1]
        stage1_moles_H2 = baseline_results['results']['moles_H2'][:perturb_index + 1]
        stage1_moles_NH3 = baseline_results['results']['moles_NH3'][:perturb_index + 1]
        stage1_conc_N2 = baseline_results['results']['conc_N2'][:perturb_index + 1]
        stage1_conc_H2 = baseline_results['results']['conc_H2'][:perturb_index + 1]
        stage1_conc_NH3 = baseline_results['results']['conc_NH3'][:perturb_index + 1]
        stage1_T = np.full_like(stage1_time, T_start)

        # Stage 2: During Perturbation
        stage2_initial_moles = (stage1_moles_N2[-1], stage1_moles_H2[-1], stage1_moles_NH3[-1])
        t_span_2 = (self.t_start, self.t_end)
        
        # select the correct ODE model based on perturbation type...
        if self.perturbation_type == 'volume':
            ode_func_2 = self._ramped_volume_model
            stage3_V = self.V_end # final volume for Stage 3
            T = self.baseline_system.T # temperature unchanged
        elif self.perturbation_type == 'injection':
            ode_func_2 = self._injection_model
            stage3_V = self.V_start # volume is unchanged for Stage 3
            T = self.baseline_system.T # temperature unchanged as well
        elif self.perturbation_type == 'temperature':
            ode_func_2 = self._ramped_temperature_model
            stage3_V = self.V_start # volume is unchanged for Stage 3 as well
            T = self.T_end # final temperature for Stage 3
            
        else:
            raise ValueError("Invalid perturbation type specified.")

        # run the solver for Stage 2...
        time_points_2 = np.linspace(self.t_start, self.t_end, 15000)
        stage2_solution = solve_ivp(ode_func_2, t_span_2, stage2_initial_moles, dense_output=True, method='LSODA', rtol=1e-6, atol=1e-9, t_eval=time_points_2)
        stage2_time = stage2_solution.t
        stage2_moles_vs_time = stage2_solution.y
        stage2_moles_N2 = stage2_moles_vs_time[0]
        stage2_moles_H2 = stage2_moles_vs_time[1]
        stage2_moles_NH3 = stage2_moles_vs_time[2]

        # calculate concentrations for Stage 2...
        if self.perturbation_type == 'volume':
            stage2_V_vs_time = np.interp(stage2_time, [self.t_start, self.t_end], [self.V_start, self.V_end])
            stage2_conc_N2 = stage2_moles_N2 / stage2_V_vs_time
            stage2_conc_H2 = stage2_moles_H2 / stage2_V_vs_time
            stage2_conc_NH3 = stage2_moles_NH3 / stage2_V_vs_time
            stage2_T = np.full_like(stage2_time, T_start)

        elif self.perturbation_type == 'temperature':
            stage2_T = np.interp(stage2_time, [self.t_start, self.t_end], [T_start, self.T_end])
            stage2_conc_N2 = stage2_moles_N2 / self.V_start
            stage2_conc_H2 = stage2_moles_H2 / self.V_start
            stage2_conc_NH3 = stage2_moles_NH3 / self.V_start

        else: # perturbation_type == 'injection'
            stage2_conc_N2 = stage2_moles_N2 / self.V_start
            stage2_conc_H2 = stage2_moles_H2 / self.V_start
            stage2_conc_NH3 = stage2_moles_NH3 / self.V_start
            stage2_T = np.full_like(stage2_time, T_start)
        
        # Stage 3: Post-Perturbation (Relaxation to New Equilibrium)
        stage3_initial_moles = (stage2_moles_N2[-1], stage2_moles_H2[-1], stage2_moles_NH3[-1])
        
        # to simulate stage 3, we create a new, temporary ChemicalSystem object that represents the state of the system at the start of stage 3.
        stage3_system = ChemicalSystem(
            initial_moles_N2 = stage3_initial_moles[0], initial_moles_H2 = stage3_initial_moles[1], initial_moles_NH3 = stage3_initial_moles[2],
            V = stage3_V, T = T)
        stage3_system.run_simulation()

        # extract results, but shift the time array to start at t_end...
        stage3_results = stage3_system.results
        stage3_time = stage3_results['time'] + self.t_end
        stage3_moles_N2 = stage3_results['results']['moles_N2']
        stage3_moles_H2 = stage3_results['results']['moles_H2']
        stage3_moles_NH3 = stage3_results['results']['moles_NH3']
        stage3_conc_N2 = stage3_results['results']['conc_N2']
        stage3_conc_H2 = stage3_results['results']['conc_H2']
        stage3_conc_NH3 = stage3_results['results']['conc_NH3']
        stage3_T = np.full_like(stage3_time, T)

        # combine the data from all three stages, slicing to avoid duplicate time points...
        combined_time = np.concatenate((stage1_time[:-1], stage2_time[:-1], stage3_time))
        combined_moles_N2 = np.concatenate((stage1_moles_N2[:-1], stage2_moles_N2[:-1], stage3_moles_N2))
        combined_moles_H2 = np.concatenate((stage1_moles_H2[:-1], stage2_moles_H2[:-1], stage3_moles_H2))
        combined_moles_NH3 = np.concatenate((stage1_moles_NH3[:-1], stage2_moles_NH3[:-1], stage3_moles_NH3))
        combined_conc_N2 = np.concatenate((stage1_conc_N2[:-1], stage2_conc_N2[:-1], stage3_conc_N2))
        combined_conc_H2 = np.concatenate((stage1_conc_H2[:-1], stage2_conc_H2[:-1], stage3_conc_H2))
        combined_conc_NH3 = np.concatenate((stage1_conc_NH3[:-1], stage2_conc_NH3[:-1], stage3_conc_NH3))
        combined_T = np.concatenate((stage1_T[:-1], stage2_T[:-1], stage3_T))
        
        # 1. Construct the Volume Array...
        stage1_V_array = np.full_like(stage1_time[:-1], self.V_start)
        if self.perturbation_type == 'volume':
             # for volume perturbation, we use the interpolated array calculated earlier...
             stage2_V_array = stage2_V_vs_time[:-1]
             stage3_V_array = np.full_like(stage3_time, self.V_end)
        else:
             # for others, volume is constant at V_start...
             stage2_V_array = np.full_like(stage2_time[:-1], self.V_start)
             stage3_V_array = np.full_like(stage3_time, self.V_start)

        combined_V = np.concatenate((stage1_V_array, stage2_V_array, stage3_V_array))

        # 2. Call the engine to get dn/dt arrays...
        dn_dt_arrays = self.baseline_system.get_rates_at_state(
            combined_moles_N2, combined_moles_H2, combined_moles_NH3, combined_V, combined_T)
        
        # 3. Convert dn/dt to Rate (mol dm^-3 s^-1)...
        combined_rates_N2 = dn_dt_arrays[0] / combined_V

        # 4. Calculate split rates for display using static Arrhenius...
        k_f_vs_time = ChemicalSystem._calculate_arrhenius_k(
            combined_T, self.baseline_system.T_ref, self.baseline_system.Ea_f, self.baseline_system.k_f_ref)
        k_r_vs_time = ChemicalSystem._calculate_arrhenius_k(
            combined_T, self.baseline_system.T_ref, self.baseline_system.Ea_r, self.baseline_system.k_r_ref)

        forward_rates = k_f_vs_time * combined_conc_N2 * combined_conc_H2**3
        reverse_rates = k_r_vs_time * combined_conc_NH3**2
        
        Qc_vs_time = np.divide(
            combined_conc_NH3**2, (combined_conc_N2 * combined_conc_H2**3),
            out=np.full_like(combined_conc_N2, np.nan),
            where=((combined_conc_N2 * combined_conc_H2**3) != 0)) 

        Kc_vs_time = k_f_vs_time / k_r_vs_time
        
        # final equilibrium values are the last point of the combined data...
        conc_N2_eq = combined_conc_N2[-1]
        conc_H2_eq = combined_conc_H2[-1]
        conc_NH3_eq = combined_conc_NH3[-1]
        n_total_eq = combined_moles_N2[-1] + combined_moles_H2[-1] + combined_moles_NH3[-1]
        Kc = (conc_NH3_eq**2) / (conc_N2_eq * conc_H2_eq**3) if (conc_N2_eq * conc_H2_eq**3) != 0 else None 
        Kc_final = Kc_vs_time[-1]

        # pressure and Kp calculations...
        combined_n_total = combined_moles_N2 + combined_moles_H2 + combined_moles_NH3
        
        # 1. Ideal Pressure.
        combined_P_ideal = (combined_n_total * R_ATM_L * combined_T) / combined_V
        
        # 2. Real Pressure (delegate to the baseline system's method).
        combined_P_real = self.baseline_system.calculate_pressure(
            combined_moles_N2, combined_moles_H2, combined_moles_NH3, combined_V, combined_T)

        # 3. Partial Pressures (using Real Pressure).
        # avoid division by zero in mole fractions if n_total is somehow 0 (unlikely but for safety)
        with np.errstate(divide='ignore', invalid='ignore'):
            # calculating the partial pressures...
            pp_N2 = (combined_moles_N2 / combined_n_total) * combined_P_real
            pp_H2 = (combined_moles_H2 / combined_n_total) * combined_P_real
            pp_NH3 = (combined_moles_NH3 / combined_n_total) * combined_P_real
            
            # 4. Qp (Reaction Quotient in Pressure), Qp = p_NH3^2 / (p_N2 * p_H2^3).
            Qp_vs_time = np.divide(
                pp_NH3**2, (pp_N2 * pp_H2**3),
                out=np.full_like(combined_P_real, np.nan),
                where=((pp_N2 * pp_H2**3) != 0))

        # 5. Kp (Thermodynamic Equilibrium Constant).
        # we can calculate the dynamic Kp based on the dynamic T: Kp = Kc * (RT)^delta_n
        # delta_n = moles_products - moles_reactants = 2 - 4 = -2
        Kc_dynamic = k_f_vs_time / k_r_vs_time
        Kp_dynamic = Kc_dynamic * (R_ATM_L * combined_T)**(-2)

        P_ideal_eq = combined_P_ideal[-1]
        P_real_eq = combined_P_real[-1]
        Kp_final = Kp_dynamic[-1]
        
        re_equilibration_duration = stage3_system.results["equilibrium"]["time"]
        t_new_eq = self.t_end + re_equilibration_duration

        print(f"Time to re-establish equilibrium after perturbation is {re_equilibration_duration:.4f} s.")

        # store the final, structured dictionary in self.results...
        self.results = {
            "time": combined_time,
            "initial_conditions": {"V": stage3_V, "T": T_start, "n_N2": combined_moles_N2[0], "n_H2": combined_moles_H2[0], "n_NH3": combined_moles_NH3[0]},
            "results": {"moles_N2": combined_moles_N2, "moles_H2": combined_moles_H2, "moles_NH3": combined_moles_NH3,
                "conc_N2": combined_conc_N2, "conc_H2": combined_conc_H2, "conc_NH3": combined_conc_NH3,
                "rates_N2": combined_rates_N2, "forward_rates": forward_rates, "reverse_rates": reverse_rates,
                        "Qc_vs_time": Qc_vs_time, "Kc_vs_time": Kc_vs_time,
                       "P_ideal": combined_P_ideal, "P_real": combined_P_real, "Qp_vs_time": Qp_vs_time, "Kp_vs_time": Kp_dynamic},
            "equilibrium": {"time": t_new_eq, "T": T, "conc_N2": conc_N2_eq, "conc_H2": conc_H2_eq, "conc_NH3": conc_NH3_eq,
                            "P_ideal": P_ideal_eq, "P_real": P_real_eq, "Kc": Kc_final, "Kp": Kp_final}}

The '```previous_system_run```' variable is used to store the results of the last baseline simulation, enabling comparison.

In [5]:
previous_system_run = None

## 4. Core Logic
This section contains the main event-handler functions that are triggered by user interactions. 

### 4.1 Display generation ('```generate_plot_and_table```')...

This is responsible for the visual output. It takes one or two complete simulation datasets and generate a 'matplotlib' plot and a 'pandas' comparison table. It will draw a vertical marker if a perturbation time is provided. This approach ensures a consistent look for all outputs and prevents code duplication.

It aims to solve several key challenges in data visualisation:
- __Handling large dynamic range:__ reaction rates can vary by many orders of magnitude. To visualise both the rapid initial rates and the near-zero rates at equilibrium, the display is split into two vertically-stacked plots. The top plot shows concentrations on a linear scale, while the bottom plot displays rates on a logarithmic scale.
- __Multi-scale time analysis:__ a simple plot cannot effectively show both a long-term simulation (e.g., >100,000s) and the fine detail of a short perturbation event (e.g., 20-21s). The function therefore generates two side-by-side views:
  - An *Overview Plot* showing the entire simulation from t=0 to the final equilibrium.
  - A *Detail Plot* which automatically zooms in on the perturbation window, providing a magnified view of the system's immediate response to stress.

- __Meaningful comparison:__ to ensure a fair comparison between a baseline and a perturbed run, the results of the ```previous_run``` are *extrapolated*. If the current simulation is longer, the previous run's final equilibrium state is extended as a horizontal line, representing the state it would have maintained if left unperturbed.
- __Axis scaling:__ the plot's time axis is always scaled to the duration of the current run.

Several metrics, such as absolute net rate ($d[N_2]/dt$) are plotted as visual indicators of equilibrium.

In [6]:
def generate_plot_and_table(current_data, previous_data, perturbation_window = None):
    """This generates and displays all visual output for a simulation run. This function takes one or two simulation results dictionaries and produces
    a matplotlib plot as well as a pandas DataFrame comparing key metrics."""
    
    #--- Data Extraction ---
    time = current_data["time"]
    V = current_data["initial_conditions"]["V"]
    conc_N2 = current_data["results"]["conc_N2"]
    conc_H2 = current_data["results"]["conc_H2"]
    conc_NH3 = current_data["results"]["conc_NH3"]
    rates_N2 = current_data["results"]["rates_N2"]
    equilibrium_time = current_data["equilibrium"]["time"]
    forward_rates = current_data["results"]["forward_rates"]
    reverse_rates = current_data["results"]["reverse_rates"]
    
    y_max_conc = 1.2 * max(np.nanmax(conc_N2), np.nanmax(conc_H2), np.nanmax(conc_NH3))
    
    if previous_data is None and equilibrium_time > 0: # this indicates a fresh baseline run
        new_slider_max = equilibrium_time
        # calculate a step size that is roughly 1/1000th of the range, but at least 1.0...
        new_step = max(1.0, round(new_slider_max / 1000))

        # update the shared perturbation time slider
        t_perturb_slider.max = new_slider_max
        t_perturb_slider.step = new_step
        t_perturb_slider.value = (int(new_slider_max * 0.4), int(new_slider_max * 0.5))
        
    # --- Table Generation---
    if previous_data is not None:
        pr = previous_data
        # helper lambda for formatting...
        fmt = lambda x: f"{x:.6f}" if x is not None else "N/A"
        fmt_p = lambda x: f"{x:.2f}" if x is not None else "N/A"
        fmt_t = lambda x: f"{x:,.4f}" if x is not None else "N/A" # for time
        
        table_data = {
            'Info': [r'Volume (dm^3)', r'Temperature (K)', 'Eq. Time (s)', 
                     r'$[N_2]$eq', r'$[H_2]$eq', r'$[NH_3]$eq',
                     'P(Ideal)/atm', 'P(Real)/atm', 'Kc', 'Kp'],
            'Previous Run': [
                f"{pr['initial_conditions']['V']:.2f}", f"{pr['equilibrium']['T']:.1f}",  fmt_t(pr['equilibrium']['time']),
                f"{pr['equilibrium']['conc_N2']:.4f}", f"{pr['equilibrium']['conc_H2']:.4f}", f"{pr['equilibrium']['conc_NH3']:.4f}",
                fmt_p(pr['equilibrium'].get('P_ideal')), fmt_p(pr['equilibrium'].get('P_real')), 
                fmt(pr['equilibrium'].get('Kc')), fmt(pr['equilibrium'].get('Kp'))
            ],
            'Current Run': [
                f"{V:.2f}", f"{current_data['equilibrium']['T']:.1f}", fmt_t(equilibrium_time),
                f"{current_data['equilibrium']['conc_N2']:.4f}", f"{current_data['equilibrium']['conc_H2']:.4f}", f"{current_data['equilibrium']['conc_NH3']:.4f}",
                fmt_p(current_data['equilibrium'].get('P_ideal')), fmt_p(current_data['equilibrium'].get('P_real')),
                fmt(current_data['equilibrium'].get('Kc')), fmt(current_data['equilibrium'].get('Kp'))
            ]}
        df = pd.DataFrame(table_data)
        display(df.style.set_table_styles([{'selector': 'th, td', 'props': [('text-align', 'center')]}]).hide(axis="index"))
    else:
        table_data = {'Info': [r'Volume ($dm^3$)', r'Temperature (K)', r'Initial $N_2$ Moles', r'Initial $H_2$ Moles', r'Initial $NH_3$ Moles', 'Eq. Time (s)', r'$[N_2]$eq', r'$[H_2]$eq', r'$[NH_3]$eq',
                               'Ideal Pressure (atm)', 'Real Pressure (atm)', 'Kc', 'Kp'],
                      'Current Run': [f"{V:.2f}", f"{current_data['equilibrium']['T']:.1f}", f"{current_data['initial_conditions']['n_N2']:.2f}", f"{current_data['initial_conditions']['n_H2']:.2f}", f"{current_data['initial_conditions']['n_NH3']:.2f}", f"{current_data['equilibrium']['time']:,.4f}" if current_data['equilibrium']['time'] else 'N/A',
                                      f"{current_data['equilibrium']['conc_N2']:.4f}", f"{current_data['equilibrium']['conc_H2']:.4f}", f"{current_data['equilibrium']['conc_NH3']:.4f}",
                                      f"{current_data['equilibrium']['P_ideal']:.2f}", f"{current_data['equilibrium']['P_real']:.2f}",
                                      f"{current_data['equilibrium']['Kc']:.4f}" if current_data['equilibrium']['Kc'] is not None else "N/A", f"{current_data['equilibrium']['Kp']:.4f}" if current_data['equilibrium']['Kc'] is not None else "N/A"]}
        df = pd.DataFrame(table_data)
        print("Baseline Simulation Results:")
        display(df.style.set_table_styles([{'selector': 'th, td', 'props': [('text-align', 'center')]}]).hide(axis="index"))

    # prepare the previous_data for plotting by extending its arrays if necessary...
    if previous_data is not None:
        pr_time_plot = previous_data['time']
        pr_conc_N2_plot = previous_data['results']['conc_N2']
        pr_conc_H2_plot = previous_data['results']['conc_H2']
        pr_conc_NH3_plot = previous_data['results']['conc_NH3']
        pr_fwd_rates_plot = previous_data['results']['forward_rates']
        pr_rev_rates_plot = previous_data['results']['reverse_rates']
        pr_net_rates_plot = np.abs(previous_data['results']['rates_N2'])
        
        old_end_time = previous_data['time'][-1]
        new_end_time = time[-1]

        if new_end_time > old_end_time: # new run takes longer to reach equilibrium than the baseline run
            pr_eq = previous_data['equilibrium']
            pr_fwd_rate_eq = previous_data['results']['forward_rates'][-1]
            pr_rev_rate_eq = previous_data['results']['reverse_rates'][-1]

            # create the new, extended arrays...
            pr_time_plot = np.concatenate((previous_data['time'], [new_end_time]))
            pr_conc_N2_plot = np.concatenate((previous_data['results']['conc_N2'], [pr_eq['conc_N2']]))
            pr_conc_H2_plot = np.concatenate((previous_data['results']['conc_H2'], [pr_eq['conc_H2']]))
            pr_conc_NH3_plot = np.concatenate((previous_data['results']['conc_NH3'], [pr_eq['conc_NH3']]))
            pr_fwd_rates_plot = np.concatenate((previous_data['results']['forward_rates'], [pr_fwd_rate_eq]))
            pr_rev_rates_plot = np.concatenate((previous_data['results']['reverse_rates'], [pr_rev_rate_eq]))
            pr_net_rates_plot = np.concatenate((pr_net_rates_plot, [1e-12])) # append a tiny number (effectively 0) for the equilibrium net rate

    # --- PLOTTING SETUP ---
    fig, axs = plt.subplots(2, 2, figsize=(20, 10), sharex='col')
    ax1, ax_rates = axs[0, 0], axs[1, 0]
    ax1_d, ax_rates_d = axs[0, 1], axs[1, 1]
    
    ax1_d.set_visible(False)
    ax_rates_d.set_visible(False)
    
    conc_handles, rate_handles, thermo_handles, marker_handles = [], [], [], []
    
    # --- PLOT 1: FULL SIMULATION OVERVIEW (left column) ---
    ax1.set_title("Full Simulation Overview")
    line, = ax1.plot(time, conc_N2, label=r"$[N_2]$", color="blue")
    conc_handles.append(line)
    line, = ax1.plot(time, conc_H2, label=r"$[H_2]$", color="red")
    conc_handles.append(line)
    line, = ax1.plot(time, conc_NH3, label=r"$[NH_3]$", color="brown")
    conc_handles.append(line)

    if previous_data is not None:
        ax1.plot(pr_time_plot, pr_conc_N2_plot, color="#ADD8E6", alpha=0.5)
        ax1.plot(pr_time_plot, pr_conc_H2_plot, color="#F08080", alpha=0.5)
        ax1.plot(pr_time_plot, pr_conc_NH3_plot, color="#C4A484", alpha=0.5)
        ax1.plot(pr_time_plot, pr_fwd_rates_plot, color="#90EE90", alpha=0.5)
        ax1.plot(pr_time_plot, pr_rev_rates_plot, color="#FFDBBB", alpha=0.5)

    if perturbation_window is not None:
        t_start, t_end = perturbation_window
        line = ax1.axvline(x=t_start, color='black', linestyle='--', label='Perturbation Window', alpha=1, lw=0.5)
        marker_handles.append(line)
        ax1.axvline(x=t_end, color='black', linestyle='--', alpha=1, lw=0.5)
        ax_rates.axvline(x=t_start, color='black', linestyle='--', alpha=1, lw=0.5)
        ax_rates.axvline(x=t_end, color='black', linestyle='--', alpha=1, lw=0.5)

    line = ax1.axvline(x=equilibrium_time, color='black', linestyle='-', label='Equilibrium Reached', alpha=0.6, lw=1)
    marker_handles.append(line)
    ax_rates.axvline(x=equilibrium_time, color='black', linestyle='-', alpha=0.6, lw=1)
    
    ax1.set_ylabel(r"Concentration / mol $dm^{-3}$")
    ax1.grid(True, linestyle='--', alpha=0.7, which='major')
    ax1.grid(True, linestyle=':', alpha=0.4, which='minor')
    ax1.minorticks_on()
    ax1.set_ylim(0, y_max_conc)
    ax1.set_xlim(left=0, right=time[-1])
    ax1.tick_params(axis='x', labeltop=True)

    line, = ax_rates.plot(time, forward_rates, label='Forward Rate', color="green", linestyle="--", lw=1.5, alpha=0.8)
    rate_handles.append(line)
    line, = ax_rates.plot(time, reverse_rates, label='Reverse Rate', color="orange", linestyle="--", lw=1.5, alpha=0.8)
    rate_handles.append(line)
    line, = ax_rates.plot(time, np.abs(rates_N2), label='|Net Rate|', color="black", linestyle=":", lw=1.2, alpha=0.8)
    rate_handles.append(line)
    
    ax_rates.set_yscale('log')
    ax_rates.set_ylim(bottom=1e-9)
    ax_rates.set_ylabel(r"Rate / mol $dm^{-3} s^{-1}$")
    ax_rates.grid(True, linestyle='--', alpha=0.7, which='major')
    ax_rates.grid(True, linestyle=':', alpha=0.4, which='minor')
    ax_rates.minorticks_on()
    ax_rates.set_xlabel("Time /s")
    
    ax2 = ax_rates.twinx()
    ax2.set_yscale('log') # log scale for the thermodynamics axis

    # plot Qc and Kp
    Qc = current_data["results"]["Qc_vs_time"]
    if "Kc_vs_time" in current_data["results"]:
        Kc_vs_time = current_data["results"]["Kc_vs_time"]
        line, = ax2.plot(time, Kc_vs_time, color='#813cba', linestyle='-.', label='Kc', alpha=0.7, lw=1)
        thermo_handles.append(line)
    else:
        if current_data["equilibrium"]["Kc"] is not None:
            line = ax2.axhline(y=current_data["equilibrium"]["Kc"], color='#813cba', linestyle='-.', label='Kc', alpha=0.7, lw=1)
            thermo_handles.append(line)
    line, = ax2.plot(time, Qc, label='Qc', color='#c159de', linestyle='-.', lw=1)
    thermo_handles.append(line)
    ax2.set_ylabel(r"Q / K (Log Scale)", color='purple')
    ax2.tick_params(axis='y', labelcolor='purple')

    # plot Qp and Kp
    Qp = current_data["results"]["Qp_vs_time"]
    Kp_vs_time = current_data["results"]["Kp_vs_time"]
    if Kp_vs_time is not None and Qp is not None:
        line, = ax2.plot(time, Kp_vs_time, color='teal', linestyle='--', label='Kp (Ideal Theory)', alpha=0.6, lw=1)
        thermo_handles.append(line)
        line, = ax2.plot(time, Qp, label='Qp', color='cyan', linestyle='--', alpha=0.6, lw=1)
        thermo_handles.append(line)

        real_kp_value = Qp[-1]
        line = ax2.axhline(y=real_kp_value, color='teal', linestyle=':', alpha=0.4, label='Kp (Observed)')
        thermo_handles.append(line)

    # --- PLOT 2: PERTURBATION DETAIL (right column) ---
    if perturbation_window is not None:
        ax1_d.set_visible(True)
        ax_rates_d.set_visible(True)
        ax1_d.set_title("Perturbation Detail (Zoomed)")

        ax1_d.plot(time, conc_N2, color="blue")
        ax1_d.plot(time, conc_H2, color="red")
        ax1_d.plot(time, conc_NH3, color="brown")
        ax_rates_d.plot(time, forward_rates, color="green", linestyle="--")
        ax_rates_d.plot(time, reverse_rates, color="orange", linestyle="--")
        ax_rates_d.plot(time, np.abs(rates_N2), color="black", linestyle=":")

        if previous_data is not None:
            ax1.plot(pr_time_plot, pr_conc_N2_plot, color="#ADD8E6", alpha=0.5)
            ax1.plot(pr_time_plot, pr_conc_H2_plot, color="#F08080", alpha=0.5)
            ax1.plot(pr_time_plot, pr_conc_NH3_plot, color="#C4A484", alpha=0.5)
            ax1.plot(pr_time_plot, pr_fwd_rates_plot, color="#90EE90", alpha=0.5)
            ax1.plot(pr_time_plot, pr_rev_rates_plot, color="#FFDBBB", alpha=0.5)
            ax1.plot(pr_time_plot, pr_net_rates_plot, color="grey", alpha=0.3, linestyle=":")

        # Thermodynamics Axis....
        ax2_d = ax_rates_d.twinx()
        ax2_d.set_yscale('log') # log scale here as well
        ax2_d.plot(time, Qc, color='#c159de', linestyle='-.')
        if "Kc_vs_time" in current_data["results"]:
             ax2_d.plot(time, current_data["results"]["Kc_vs_time"], color='#813cba', linestyle='-.')
        else:
             ax2_d.axhline(y=current_data["equilibrium"]["Kc"], color='#813cba', linestyle='-.')

        if Kp_vs_time is not None and Qp is not None:
             ax2_d.plot(time, Kp_vs_time, color='teal', linestyle='--')
             ax2_d.plot(time, Qp, color='cyan', linestyle='--')

        t_start, t_end = perturbation_window
        ax1_d.axvline(x=t_start, color='black', linestyle='--')
        ax1_d.axvline(x=t_end, color='black', linestyle='--')
        ax_rates_d.axvline(x=t_start, color='black', linestyle='--')
        ax_rates_d.axvline(x=t_end, color='black', linestyle='--')
        ax1_d.axvline(x=equilibrium_time, color='black', linestyle='-')
        ax_rates_d.axvline(x=equilibrium_time, color='black', linestyle='-')
        
        ax1_d.set_ylabel(r"Concentration / mol $dm^{-3}$")
        ax_rates_d.set_ylabel(r"Rate / mol $dm^{-3} s^{-1}$")
        ax2_d.set_ylabel(r"Q / K", color='purple')
        ax2_d.tick_params(axis='y', labelcolor='purple')
        ax_rates_d.set_xlabel("Time /s")
        ax_rates_d.set_yscale('log')
        ax_rates_d.set_ylim(bottom=1e-9)
        ax1_d.grid(True, linestyle='--', which='major')
        ax_rates_d.grid(True, linestyle='--', which='major')
        ax1_d.grid(True, linestyle=':', alpha=0.7, which='minor')
        ax_rates_d.grid(True, linestyle=':', alpha=0.7, which='minor')
        ax1_d.minorticks_on()
        ax_rates_d.minorticks_on()
        ax1_d.set_ylim(0, y_max_conc)

        # creating a dynamic time axis for the perturbation detail plot
        perturb_duration = t_end - t_start
        margin_before = perturb_duration * 0.5 
        margin_after = perturb_duration * 1.8
        
        x_min_detail = max(0, t_start - margin_before)
        x_max_detail = min(time[-1], t_end + margin_after)
        current_width = x_max_detail - x_min_detail
        min_view_width = 20.0
        
        if current_width < min_view_width:
            center_point = (t_start + t_end) / 2
            half_width = min_view_width / 2
            x_min_detail = max(0, center_point - half_width)
            # ensure we don't look past the end of data
            x_max_detail = min(time[-1], x_min_detail + min_view_width) 

        ax1_d.set_xlim(x_min_detail, x_max_detail)
        ax1_d.tick_params(axis='x', labeltop=True)

    # --- FINAL LEGEND AND LAYOUT ---
    leg1 = fig.legend(handles=conc_handles + marker_handles, loc='upper left', bbox_to_anchor=(0.12, 0.95), title="Concentrations")
    leg2 = fig.legend(handles=rate_handles, loc='center left', bbox_to_anchor=(0.12, 0.5), title="Rates")
    leg3 = fig.legend(handles=thermo_handles, loc='lower left', bbox_to_anchor=(0.12, 0.15), title="Equilibrium")
    
    fig.subplots_adjust(hspace=0, left=0.22)
    
    plt.show()

### 4.2 UI Controllers...

These functions act as controllers, connecting the user interface (UI) widgets to the backend simulation engine. They are not responsible for performing any chemical calculations themselves. Instead, their role is to:
1. Listen for user events (button clicks).
2. Read the relevant parameters (initial concentrations, perturbation windows) from the UI controls.
3. Instantiate the appropriate backend object (```ChemicalSystem``` for a baseline run or ```PerturbationSimulation``` for a perturbation).
4. Issue a command to that object (e.g., ```.run_simulation()``` or ```.runperturbation()```).
5. Retrieve the final, processed results dictionary from the object.
6. Pass this data onto ```generate_plot_and_table``` function for visualisation.
This keeps the UI logic clean and delegates all complex work to the specialised objects. 

In [7]:
def run_baseline_controller(button):
    """This acts as the main controller when the 'Run Simulation' button is clicked.
    
    This function reads parameters from the UI, creates a ChemicalSystem object, tells it to run its simulation, and then passes the processed results
    to the display function. It also saves the state for the next run."""
    
    global previous_system_run 
    with output:
        output.clear_output(wait=True)

        # 1. Read initial conditions from the UI sliders...
        V = V_slider.value
        T = T_slider.value
        initial_moles_N2 = N2_slider.value * V
        initial_moles_H2 = H2_slider.value * V
        initial_moles_NH3 = NH3_slider.value * V

        # some input validation
        if initial_moles_N2 == 0 and initial_moles_H2 == 0 and initial_moles_NH3 == 0:
            print("Error: Initial moles for all the species cannot be zero. Please set an initial concentration for at least one species.")
            return

        # 2. Instantiate the ChemicalSystem object...
        current_system = ChemicalSystem(
            initial_moles_N2 = initial_moles_N2, initial_moles_H2 = initial_moles_H2, initial_moles_NH3 = initial_moles_NH3,
            V = V, T = T)

        # 3. Run the simulation and automatically process the results...
        current_system.run_simulation()

        # 4. Prepare the data dictionaries for the plotting function...
        current_data = current_system.results    # extract the .results dictionary from our objects
        previous_data = previous_system_run.results if previous_system_run else None

        # 5. Call the shared plot and table generation function...
        generate_plot_and_table(current_data, previous_data)

        # 6. Save the state for the next run by storing the entire object...
        previous_system_run = copy.deepcopy(current_system)

        # 7. Enable perturbation controls for the next action...
        t_perturb_slider.disabled = False
        perturb_volume_button.disabled = False
        inject_species_button.disabled = False
        N2_injection_rate_widget.disabled = False
        H2_injection_rate_widget.disabled = False
        NH3_injection_rate_widget.disabled = False
        perturb_temperature_button.disabled = False

In [8]:
def apply_perturbation(button, perturbation_type):
    """This acts as the controller when a perturbation button is clicked.

    This function reads perturbation parameters from the UI, creates a PerturbationSimulation object to manage the process, tells it to run,
    and then passes the final results to the display function."""
    
    global previous_system_run
    with output:
        output.clear_output(wait=True)

        if previous_system_run is None:
            print("Please run a baseline simulation first.")
            return

        # 1. Read perturbation parameters from the UI...
        if perturbation_type == 'volume':
            print("Applying volume perturbation...")
            t_start, t_end = t_perturb_slider.value
            V_end = V_slider.value
            if V_end == previous_system_run.volume:
                print("Warning: target volume is the same as current volume.")
            # for this perturbation type, injection rates are zero
            inject_N2 = 0.0
            inject_H2 = 0.0
            inject_NH3 = 0.0
            # temperature is constant
            T_end = previous_system_run.T
            
        elif perturbation_type == 'injection':
            print("Applying injection perturbation...")
            t_start, t_end = t_perturb_slider.value
            inject_N2 = N2_injection_rate_widget.value
            inject_H2 = H2_injection_rate_widget.value
            inject_NH3 = NH3_injection_rate_widget.value
            # for this perturbation type, the end volume and temperature is the same as the start
            V_end = previous_system_run.volume
            T_end = previous_system_run.T
            
        elif perturbation_type == 'temperature':
            print("Applying temperature perturbation...")
            t_start, t_end = t_perturb_slider.value
            T_end = T_slider.value
            if T_end == previous_system_run.T:
                print("Warning: target temperature is the same as current temperature.")  
            # for this perturbation type, the end voluem is the same as the start and the injection rates are zero
            V_end = previous_system_run.volume
            inject_N2 = 0.0
            inject_H2 = 0.0
            inject_NH3 = 0.0
            
        else:
            return # should not happen

        # 2. Instantiate the process manager class...
        perturbation_sim = PerturbationSimulation(
            baseline_system = previous_system_run,
            perturbation_type = perturbation_type,
            t_start = t_start,
            t_end = t_end,
            V_end = V_end,
            T_end = T_end,
            injection_rate_N2 = inject_N2,
            injection_rate_H2 = inject_H2,
            injection_rate_NH3 = inject_NH3)

        # 3. Run the entire three-stage simulation...
        perturbation_sim.run_perturbation()

        # 4. Get the final results from the simulation object...
        combined_data = perturbation_sim.results
        previous_data = previous_system_run.results # get baseline data for comparison

        # 5. Call the display function with the new and old data...
        generate_plot_and_table(
            current_data = combined_data,
            previous_data = previous_data,
            perturbation_window = (t_start, t_end))

### 4.4 Utility functions...

This cell contains smaller helper functions, such as the event handler for the 'Clear' button.

In [9]:
def clear_previous_run_data(button):
    global previous_system_run
    previous_system_run = None
    t_perturb_slider.disabled = True
    perturb_volume_button.disabled = True
    inject_species_button.disabled = True
    N2_injection_rate_widget.disabled = True
    H2_injection_rate_widget.disabled = True
    NH3_injection_rate_widget.disabled = True
    perturb_temperature_button.disabled = True
    t_perturb_slider.max = 100
    with output:
        output.clear_output(wait = True)
        print("Previous run data has been cleared.")

## 5. User Interface (UI)
This cell is where the widgets are defined and configured. This includes the sliders for setting initial conditions and the buttons for controlling the simulation.

In [10]:
# --- UI Widget Creation and Layout ---
slider_layout = widgets.Layout(width='600px')
slider_style = {'description_width': '180px'}

# --- Sliders ---
N2_slider = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.05, description=r'Initial $[N_2]$ (mol dm$^{-3}$)', continuous_update=False, layout=slider_layout, style=slider_style)
H2_slider = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.05, description=r'Initial $[H_2]$ (mol dm$^{-3}$)', continuous_update=False, layout=slider_layout, style=slider_style)
NH3_slider = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.05, description=r'Initial $[NH_3]$ (mol dm$^{-3}$)', continuous_update=False, layout=slider_layout, style=slider_style)
V_slider = widgets.FloatSlider(value=1.0, min=0.05, max=10.0, step=0.05, description=r'Volume (dm$^3$)', continuous_update=False, layout=slider_layout, style=slider_style)
T_slider = widgets.FloatSlider(value=500, min=400, max=700, step=5, description=r'Temperature (K)', continuous_update=False, layout=slider_layout, style=slider_style)

t_perturb_slider = widgets.FloatRangeSlider(value=(0.5, 0.7), min=0.05, max=100.0, step=0.01, description='Perturbation Time Range', continuous_update=False, layout=slider_layout, style=slider_style, disabled = True)

# --- Injection Rates ---
N2_injection_rate_widget = widgets.BoundedFloatText(value=0.0, min=-50, max=50, description=r'$N_2$ Injection Rate (mol/s)', disabled = True, layout=slider_layout, style=slider_style)
H2_injection_rate_widget = widgets.BoundedFloatText(value=0.0, min=-50, max=50, description=r'$H_2$ Injection Rate (mol/s)', disabled = True, layout=slider_layout, style=slider_style)
NH3_injection_rate_widget = widgets.BoundedFloatText(value=0.0, min=-50, max=50, description=r'$NH_3$ Injection Rate (mol/s)', disabled = True, layout=slider_layout, style=slider_style)

# --- Buttons ---
run_button = widgets.Button(description="Run Simulation", button_style='success')
clear_button = widgets.Button(description="Clear Previous Run", button_style='warning')
inject_species_button = widgets.Button(description="Inject Species", disabled=True)
perturb_volume_button = widgets.Button(description="Perturb System (Volume)", disabled = True)
perturb_temperature_button = widgets.Button(description="Perturb System (Temperature)", disabled = True)

button_layout = widgets.Layout(width='200px', margin='5px 10px 5px 0')
run_button.layout = button_layout
clear_button.layout = button_layout
perturb_volume_button.layout = button_layout
inject_species_button.layout = button_layout
perturb_temperature_button.layout = button_layout

## 6. Application Initialisation
This cell connects the UI widgets to the core logic by setting up the 'on_click' event listeners. Since the function apply_perturbation must know which button (Perturb Volume or Inject Species) is triggered, a labmda function is used. This lambda function calls the main apply_perturbation and also passes a custom string that tells the main function which block of logic to execute.

In [11]:
# --- Event Handling ---
run_button.on_click(run_baseline_controller)
clear_button.on_click(clear_previous_run_data)
perturb_volume_button.on_click(lambda b: apply_perturbation(b, 'volume'))
inject_species_button.on_click(lambda b: apply_perturbation(b, 'injection'))
perturb_temperature_button.on_click(lambda b: apply_perturbation(b, 'temperature'))

## 7. Dashboard
This cell displays and runs the simulation.

In [12]:
# --- Display UI ---
slider_box = widgets.VBox([N2_slider, H2_slider, NH3_slider, V_slider, T_slider, t_perturb_slider], layout=widgets.Layout(align_items='flex-start', padding='10px'))
button_box = widgets.HBox([run_button, clear_button, perturb_volume_button, perturb_temperature_button, inject_species_button])
injection_controls = widgets.VBox([N2_injection_rate_widget, H2_injection_rate_widget, NH3_injection_rate_widget])
output = widgets.Output()
app_layout = widgets.VBox([output, slider_box, injection_controls, button_box])
display(app_layout)

## 8. Model Assumptions and Simplifications

### Project Scope
The industrial Haber-Bosch process is a triumph of engineering, relying on continuous flow loops, condensers to remove product, and complex heat exchangers to manage the adiabatic temperature rise.

However, this project is a strict investigation into physical chemistry rather than engineering. By modelling the system as a closed, isothermal batch reactor, we intentionally strip away the engineering mechanisms used to "cheat" equilibrium (such as product removal). This allows us to isolate and visualise the intrinsic physicochemical properties of the reaction itself: the tug-of-war between the Arrhenius Equation (kinetics) and the Van 't Hoff Equation (thermodynamics). Some of the assumptions listed below are therefore not merely simplifications, but necessary control variables to study the fundamental behaviour of the chemical system in its purest form.

### Limitations of the Model
__Rate Laws__

The true kinetic mechanism of the iron-catalysed Haber process is extraordinarily complex.  It is a heterogeneous reaction occurring on the surface of the catalyst and is understood to proceed via a multi-step mechanism, often described by models such as the Langmuir-Hinshelwood or Temkin-Pyzhev kinetics. These models account for the adsorption of reactant gases onto active sites on the catalyst surface, the surface reaction itself, and the desorption of the product. The resulting rate laws are mathematically intricate and depend on the partial pressures of all three species in a non-trivial manner.

For the purposes of this simulation, which focuses on demonstrating the principles of dynamic equilibrium and Le Chatelier's Principle rather than replicating industrial synthesis with mechanistic precision, a simplified rate law has been adopted:

Forward Rate = $k_f[N_2][H_2]^3$

Reverse Rate = $k_r[NH_3]^2$

While not mechanistically accurate, this approximation has two key advantages. Firstly, its exponents match the stoichiometry of the overall reaction, ensuring that the model correctly represents the relationship between the consumption of reactants and the formation of product. Secondly, it provides a mathematically tractable system that robustly models the approach to a thermodynamically correct equilibrium state. The core objective (to simulate how the system responds to perturbations in concentration, volume, and temperature) is achieved effectively with this simplified kinetic model.

Although the model has this simplification, the temperature dependence of equilibrium ($K_c$) remains valid regardless of the kinetic model chosen, provided the forward/reverse $E_a$ difference equals $\Delta H$.

__Thermal Constraints__

The model treats the reaction environment as strictly isothermal ($dT/dt = 0$ unless externally perturbed). This neglects the enthalpy of reaction, which in a real, adiabatic system would increase the internal kinetic energy of the molecules, raising the system's temperature. Such a temperature rise would kinetically accelerate the reaction but thermodynamically disfavour the product formation, as the equilibrium constant ($K_c$) decreases according to the van 't Hoff equation.

__Temperature Independence of $\Delta H$__

The model treats the enthalpy of reaction ($\Delta H$) as a constant -92 kJ/mol across all temperatures. Strictily speaking, $\Delta H$ varies with temperature according to Kirchoff's Law of Thermochemistry. 

__Single Phase Behaviour__

This simulation assumes all species remain in the gas phase under all conditions. In the industrial process, the gas mixture is cooled to liquefy and separate the ammonia ($NH_3(l)$), which drives the equilibrium to the right. Since this model is focused on purely the chemistry, it simulates only the reaction system.